In [1]:
import pickle
import os
import time
import numpy as np
import random
import pandas as pd
from operator import itemgetter
from tqdm import tqdm

In [2]:
runtime = time.time()
home = 'C:./datasets'

In [3]:
DATASET_DIR = home + '/amazon'
#ORDERS_FILE = DATASET_DIR + '/Clothing_Shoes_and_Jewelry_Result_sample.csv'
ORDERS_FILE = DATASET_DIR + '/Clothing_Shoes_and_Jewelry_Result_2012_2014.csv'

'''
DATASET_USER_SESSIONS_SORTING = DATASET_DIR + '/user_sessions_sorting.pickle'
DATASET_USER_SESSIONS_RENAME = DATASET_DIR + '/user_sessions_rename.pickle'
DATASET_TRAIN_TEST_SPLIT_PAD_VALUE = DATASET_DIR + '/train_test_split_pad_value.pickle'
'''
DATASET_USER_SESSIONS_SORTING = DATASET_DIR + '/2_user_sessions.pickle'
DATASET_USER_SESSIONS_RENAME = DATASET_DIR + '/3_user_sessions_filter.pickle'
DATASET_TRAIN_TEST_SPLIT_PAD_VALUE = DATASET_DIR + '/4_train_test_split.pickle'
DATASET_TRAIN_TEST_SPLIT_PAD_VALUE_SAMPLE = DATASET_DIR + '/4_train_test_split_sample.pickle'


data_dir = DATASET_DIR + '/'
seed = 12345
NUM_NEGATIVE_SAMPLES = 100
NEGATIVE_SAMPLER_SEED = seed
num_samples=100

In [4]:
MAX_SESSION_LENGTH = 20
MINIMUM_REQUIRED_SESSIONS = 3 #如果用戶的session中的互動少於X，則刪過少的session
PAD_VALUE = 0 #補值:補0

In [5]:
def file_exists(filename):
    return os.path.isfile(filename)

def load_pickle(pickle_file):
    return pickle.load(open(pickle_file, 'rb'))

def save_pickle(data_object, data_file):
    pickle.dump(data_object, open(data_file, 'wb'))

# user_sessions_sorting

In [6]:
if not file_exists(DATASET_USER_SESSIONS_SORTING):
    user_orders = {}
    print("Reading user orders.")
    
    with open(ORDERS_FILE, 'rt', buffering=10000, encoding='utf8') as orders:
        next(orders) # skip header
        # user_id  product_id  reviewTime  unixReviewTime
        for line in orders:
            line = line.rstrip() #刪除空行
            line = line.split(',') #將每行遇到,就用''切割成value
            
            #各自賦予意義: 用戶_id / 產品_id / 瀏覽時長(原始) / 瀏覽時長(nuix)
            user_id        = line[0] 
            product_id     = line[1]
            reviewTime     = line[2]+line[3]
            unixReviewTime = line[4]
            
            #判斷 如果用戶_id不在裡面就...
            if user_id not in user_orders:
                
                #在此uesr_id的位置設立空集合
                user_orders[user_id] = [] 
                
            #將方才的2個value放入一個[]append進去，此時裡面是2維向量   
            user_orders[user_id].append([product_id, unixReviewTime]) 
            
            #這筆資料集有402093個用戶，當我key超過閥值就break，但顯然是不會超過
            if len(user_orders.keys()) >1000000 : 
                break

Reading user orders.


In [7]:
print("Sorting user orders.")
# ensure that orders are sorted for each user

for user_id in user_orders.keys():
    orders = user_orders[user_id]
    user_orders[user_id] = sorted(orders, key=lambda x: x[1]) #利用key，按照時間長短x[1]排序

Sorting user orders.


In [8]:
print("Connecting orders and users.")

results={}
results_orders={}

for user_id in user_orders.keys():
    results[user_id] = {}
    for x in user_orders[user_id]:
        if x[1] not in results[user_id]:#x[1]點擊時間
            results[user_id][x[1]]=[]

        results[user_id][x[1]].append([len(results[user_id][x[1]])+1,x[0]])


for user_id in results.keys():
    results_orders[user_id] = []
    for unixReviewTime in results[user_id].keys():
        results_orders[user_id].append(results[user_id][unixReviewTime])
    #print(results_orders[user_id])
    user_orders[user_id]=results_orders[user_id]


del results
del results_orders

Connecting orders and users.


In [9]:
print("Calculating some statistics.")
session_lengths = [0]*155
products = {}
n_sessions = 0
longest = 0
shortest = 999999
for user, orders in user_orders.items():
    n_sessions += len(orders) #計算session有幾個? 直接計算orders多長就好
    for order in orders:
        if len(order) > longest:
            longest = len(order)  #當滿足條件 就會一直被更新最長的session
        if len(order) < shortest:
            shortest = len(order) #當滿足條件 就會一直被更新最短的session
        session_lengths[len(order)] += 1 #蠻特殊的，每個session中內容多長，用len計算，並將之+1記錄到session_lengths[]符合的位置
        for x in order:
            product = x[1]
            if product not in products:
                products[product] = True #只是將與之對應位置賦予True
print("num products (labels):", len(products.keys())) #幾種商品
print("num users:", len(user_orders.keys())) #幾個user
print("num sessions:", n_sessions) #幾個session
print("shortest session:", shortest) #經累計計算後最短session
print("longest session:", longest) #經累計計算後最長session
print()
print("SESSION LENGTHS:")
for i in range(len(session_lengths)):
    print(i, session_lengths[i]) #將方才紀錄的位置show出來

save_pickle(user_orders, DATASET_USER_SESSIONS_SORTING)

Calculating some statistics.
num products (labels): 140116
num users: 402093
num sessions: 770290
shortest session: 1
longest session: 60

SESSION LENGTHS:
0 0
1 516990
2 140103
3 53164
4 25299
5 14738
6 7324
7 3375
8 1813
9 1784
10 1401
11 1362
12 471
13 287
14 269
15 242
16 130
17 150
18 163
19 351
20 284
21 187
22 87
23 48
24 43
25 22
26 14
27 16
28 36
29 37
30 29
31 24
32 12
33 8
34 7
35 3
36 5
37 1
38 3
39 1
40 2
41 0
42 1
43 0
44 1
45 0
46 0
47 0
48 2
49 0
50 0
51 0
52 0
53 0
54 0
55 0
56 0
57 0
58 0
59 0
60 1
61 0
62 0
63 0
64 0
65 0
66 0
67 0
68 0
69 0
70 0
71 0
72 0
73 0
74 0
75 0
76 0
77 0
78 0
79 0
80 0
81 0
82 0
83 0
84 0
85 0
86 0
87 0
88 0
89 0
90 0
91 0
92 0
93 0
94 0
95 0
96 0
97 0
98 0
99 0
100 0
101 0
102 0
103 0
104 0
105 0
106 0
107 0
108 0
109 0
110 0
111 0
112 0
113 0
114 0
115 0
116 0
117 0
118 0
119 0
120 0
121 0
122 0
123 0
124 0
125 0
126 0
127 0
128 0
129 0
130 0
131 0
132 0
133 0
134 0
135 0
136 0
137 0
138 0
139 0
140 0
141 0
142 0
143 0
144 0
145 0
146 0
1

# user_sessions_rename

In [10]:
if not file_exists(DATASET_USER_SESSIONS_RENAME):
    user_sessions = load_pickle(DATASET_USER_SESSIONS_SORTING)

    # Split sessions
    def split_single_session(session):  
        splitted = [session[i:i+MAX_SESSION_LENGTH] for i in range(0, len(session), MAX_SESSION_LENGTH)]
        if len(splitted[-1]) < 2:
            del splitted[-1]
        return splitted

    def perform_session_splits(sessions):
        splitted_sessions = []
        for session in sessions:
            splitted_sessions += split_single_session(session)
        return splitted_sessions

In [11]:
#print(user_sessions)
for k in user_sessions.keys():
    sessions = user_sessions[k]
    #print(sessions)
    user_sessions[k] = perform_session_splits(sessions)
    #print(user_sessions[k])

In [12]:
# Remove too short sessions 刪掉過短session
for k in user_sessions.keys():
    sessions = user_sessions[k]
    user_sessions[k] = [s for s in sessions if len(s)>1]
    #print(user_sessions[k])

In [13]:
# Remove users with too few sessions 刪掉過少的session
users_to_remove = []
for u, sessions in user_sessions.items():
    if len(sessions) < MINIMUM_REQUIRED_SESSIONS:
        users_to_remove.append(u)

for u in users_to_remove:
    del(user_sessions[u])
    
#for k in user_sessions.items():
    #print(k)

In [14]:
# Rename user_ids 用戶
if len(users_to_remove) > 0:
    nus = {}
    for k, v in user_sessions.items():
        nus[len(nus)] = user_sessions[k] 
#nus

In [15]:
# Rename labels
lab = {}
for k, v in nus.items():
    for session in v:
        for i in range(len(session)):
            if isinstance(session[i][1], str) == True:
                l = session[i][1]
                if l not in lab:
                    lab[l] = len(lab)+1
                session[i][1] = lab[l]
#nus

In [16]:
print('檢查是否有空值')
for k, v in nus.items():
    for session in v:
        if not session:
            print('Has Empty !!!!!'+ str(session))

save_pickle(nus, DATASET_USER_SESSIONS_RENAME)

檢查是否有空值


# BERT4Rec: general_preprocessing

In [17]:
dataset = load_pickle(DATASET_USER_SESSIONS_RENAME)

In [18]:
user_list_1 = []
count_timestamp = 0
for user, all_session in dataset.items():
    #print(user)
    #print(all_session)
    for session in all_session:
        count_timestamp +=1
        for action in session:
            user_list_1.append([user, count_timestamp, action[1]])
        

#print('user_list_1\n%s' %(user_list_1))

In [19]:
user_list_2 = pd.DataFrame(user_list_1) 
user_list_2.columns = ['uid', 'timestamp', 'iid']
df_rows = user_list_2

In [20]:
user_list_2

,uid,timestamp,iid
0,0,1,1
1,0,1,2
2,0,1,3
3,0,1,4
4,0,2,5
...,...,...,...
105967,9732,35046,23855
105968,9732,35047,21733
105969,9732,35047,19593
105970,9732,35048,18100


In [21]:
def cut_and_assign_sids_to_rows(rows):
    sid = 0
    uid2rows = {}
    for uid, timestamp, iid in tqdm(rows, desc="* organize uid2rows"):
        if uid not in uid2rows:
            uid2rows[uid] = []
        uid2rows[uid].append((iid, timestamp))
    rows = []
    uids = list(uid2rows.keys())
    for uid in tqdm(uids, desc="* cutting"):
        user_rows = sorted(uid2rows[uid], key=itemgetter(1))
        tba = []
        sid2count = {}
        sid += 1
        _, previous_timestamp = user_rows[0]
        for iid, timestamp in user_rows:
            '''
            if timestamp - previous_timestamp > SESSION_WINDOW:
                sid += 1
            '''
            if timestamp != previous_timestamp:
                sid += 1
            
            tba.append((uid, iid, sid, timestamp))
            sid2count[sid] = sid2count.get(sid, 0) + 1
            previous_timestamp = timestamp

        rows.extend(tba)
    return rows

In [22]:
# cut and assign sid
print("- cut and assign sid")
rows = cut_and_assign_sids_to_rows(df_rows.values)
df_rows = pd.DataFrame(rows)
df_rows.columns = ['uid', 'iid', 'sid', 'timestamp']

- cut and assign sid


* cutting: 100%|███████████████████████████████████████████████████████████████| 9733/9733 [00:00<00:00, 159549.61it/s]


In [23]:
# map uid -> uindex
print("- map uid -> uindex")
uids = set(df_rows['uid'])
uid2uindex = {uid: index for index, uid in enumerate(set(uids), start=1)}
df_rows['uindex'] = df_rows['uid'].map(uid2uindex)
df_rows = df_rows.drop(columns=['uid'])
with open(os.path.join(data_dir, 'uid2uindex.pkl'), 'wb') as fp:
    pickle.dump(uid2uindex, fp)

- map uid -> uindex


In [24]:
# map iid -> iindex
print("- map iid -> iindex")
iids = set(df_rows['iid'])
iid2iindex = {iid: index for index, iid in enumerate(set(iids), start=1)}
df_rows['iindex'] = df_rows['iid'].map(iid2iindex)
df_rows = df_rows.drop(columns=['iid'])
with open(os.path.join(data_dir, 'iid2iindex.pkl'), 'wb') as fp:
    pickle.dump(iid2iindex, fp)

- map iid -> iindex


In [25]:
# save df_rows
print("- save df_rows")
df_rows.to_pickle(os.path.join(data_dir, 'df_rows.pkl'))

- save df_rows


In [26]:
df_rows = DATASET_DIR + '/df_rows.pkl'
df_rows = load_pickle(df_rows)

In [27]:
df_rows

,sid,timestamp,uindex,iindex
0,1,1,1,1
1,1,1,1,2
2,1,1,1,3
3,1,1,1,4
4,2,2,1,5
...,...,...,...,...
105967,35046,35046,9733,23855
105968,35047,35047,9733,21733
105969,35047,35047,9733,19593
105970,35048,35048,9733,18100


In [28]:
# split train, valid, test
print("- split train, valid, test")
train_data = {}
valid_data = {}
test_data = {}
for uindex in tqdm(list(uid2uindex.values()), desc="* splitting"):
    df_user_rows = df_rows[df_rows['uindex'] == uindex].sort_values(by='timestamp')
    user_rows = list(df_user_rows[['iindex', 'sid', 'timestamp']].itertuples(index=False, name=None))
    train_data[uindex] = user_rows[:-2]
    valid_data[uindex] = user_rows[-2: -1]
    test_data[uindex] = user_rows[-1:]

# save splits
print("- save splits")
with open(os.path.join(data_dir, 'train.pkl'), 'wb') as fp:
    pickle.dump(train_data, fp)
with open(os.path.join(data_dir, 'valid.pkl'), 'wb') as fp:
    pickle.dump(valid_data, fp)
with open(os.path.join(data_dir, 'test.pkl'), 'wb') as fp:
    pickle.dump(test_data, fp)

- split train, valid, test


* splitting: 100%|████████████████████████████████████████████████████████████████| 9733/9733 [00:18<00:00, 535.86it/s]

- save splits


# BERT4Rec: random negative

In [29]:
print("do general random negative sampling")

# load materials
print("- load materials")
with open(os.path.join(data_dir, 'df_rows.pkl'), 'rb') as fp:
    df_rows = pickle.load(fp)
with open(os.path.join(data_dir, 'uid2uindex.pkl'), 'rb') as fp:
    uid2uindex = pickle.load(fp)
    user_count = len(uid2uindex)
with open(os.path.join(data_dir, 'iid2iindex.pkl'), 'rb') as fp:
    iid2iindex = pickle.load(fp)
    item_count = len(iid2iindex)

# sample random negatives
print("- sample random negatives")
ns = {}
np.random.seed(seed)
for uindex in tqdm(list(range(1, user_count + 1)), desc="* sampling"):
    seen_iindices = set(df_rows[df_rows['uindex'] == uindex]['iindex'])
    sampled_iindices = set()
    for _ in range(num_samples):
        iindex = np.random.choice(item_count) + 1
        while iindex in seen_iindices or iindex in sampled_iindices:
            iindex = np.random.choice(item_count) + 1
        sampled_iindices.add(iindex)
    ns[uindex] = list(sampled_iindices)

# save sampled random nagetives
print("- save sampled random nagetives")
with open(os.path.join(data_dir, 'ns_random.pkl'), 'wb') as fp:
    pickle.dump(ns, fp)

do general random negative sampling
- load materials
- sample random negatives


* sampling: 100%|█████████████████████████████████████████████████████████████████| 9733/9733 [00:13<00:00, 732.13it/s]

- save sampled random nagetives


# BERT4Rec: popular negative

In [30]:
print("do general popular negative sampling")

# load materials
print("- load materials")
with open(os.path.join(data_dir, 'df_rows.pkl'), 'rb') as fp:
    df_rows = pickle.load(fp)
with open(os.path.join(data_dir, 'uid2uindex.pkl'), 'rb') as fp:
    uid2uindex = pickle.load(fp)
    user_count = len(uid2uindex)

# reorder items
print("- reorder items")
reordered_iindices = list(df_rows.groupby(['iindex']).size().sort_values().index)[::-1]

# sample popular negatives
print("- sample popular negatives")
ns = {}
for uindex in tqdm(list(range(1, user_count + 1)), desc="* sampling"):
    seen_iindices = set(df_rows[df_rows['uindex'] == uindex]['iindex'])
    sampled_iindices = []
    for iindex in reordered_iindices:
        if len(sampled_iindices) == num_samples:
            break
        if iindex in seen_iindices:
            continue
        sampled_iindices.append(iindex)
    ns[uindex] = sampled_iindices

# save sampled popular nagetives
print("- save sampled popular nagetives")
with open(os.path.join(data_dir, 'ns_popular.pkl'), 'wb') as fp:
    pickle.dump(ns, fp)

do general popular negative sampling
- load materials
- reorder items
- sample popular negatives


* sampling: 100%|████████████████████████████████████████████████████████████████| 9733/9733 [00:06<00:00, 1404.16it/s]

- save sampled popular nagetives


# BERT4Rec: count stats

In [31]:
print("task: count stats")

print('\t'.join([
    "dname",
    "#user",
    "#item",
    "#row",
    "density",
    "ic_25",
    "ic_50",
    "ic_75",
    "ic_95",
    "sc_25",
    "sc_50",
    "sc_75",
    "sc_95",
    "cc_25",
    "cc_50",
    "cc_75",
    "cc_95",
]))


# load data
with open(os.path.join(data_dir, 'uid2uindex.pkl'), 'rb') as fp:
    uid2uindex = pickle.load(fp)
with open(os.path.join(data_dir, 'iid2iindex.pkl'), 'rb') as fp:
    iid2iindex = pickle.load(fp)
with open(os.path.join(data_dir, 'df_rows.pkl'), 'rb') as fp:
    df_rows = pickle.load(fp)

# get density
num_users = len(uid2uindex)
num_items = len(iid2iindex)
num_rows = len(df_rows)
density = num_rows / num_users / num_items

# get item count per user
icounts = df_rows.groupby('uindex').size().to_numpy()  # allow duplicates

# get session count per user
scounts = df_rows.groupby('uindex').agg({'sid': 'nunique'})['sid'].to_numpy()

# get item count per user-session
ccounts = df_rows.groupby(['uindex', 'sid']).size().to_numpy()

# report
print('\t'.join([
    'amazon',
    str(num_users),
    str(num_items),
    str(num_rows),
    f"{100 * density:.04f}%",
    str(int(np.percentile(icounts, 25))),
    str(int(np.percentile(icounts, 50))),
    str(int(np.percentile(icounts, 75))),
    str(int(np.percentile(icounts, 95))),
    str(int(np.percentile(scounts, 25))),
    str(int(np.percentile(scounts, 50))),
    str(int(np.percentile(scounts, 75))),
    str(int(np.percentile(scounts, 95))),
    str(int(np.percentile(ccounts, 25))),
    str(int(np.percentile(ccounts, 50))),
    str(int(np.percentile(ccounts, 75))),
    str(int(np.percentile(ccounts, 95))),
]))

task: count stats
dname	#user	#item	#row	density	ic_25	ic_50	ic_75	ic_95	sc_25	sc_50	sc_75	sc_95	cc_25	cc_50	cc_75	cc_95
amazon	9733	46958	105972	0.0232%	7	9	12	23	3	3	4	6	2	2	3	6
